In [1]:
# Q1. 지금 인기 있는 영화가 무엇인지 알려줘.
# Q2. movie ID 550에 해당하는 영화가 무엇인지 알려줘
# Q3. movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘.
import requests

base_url = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    res = requests.get(f"{base_url}/movies")
    return str(res.content)

def get_movie_details(id):
    res = requests.get(f"{base_url}/movies/{id}")
    return str(res.content)

def get_movie_credits(id):
    res = requests.get(f"{base_url}/movies/{id}/credits")
    return str(res.content)

def get_similar_movies(id):
    res = requests.get(f"{base_url}/movies/{id}/similar")
    return str(res.content)

In [2]:
from dotenv import load_dotenv
import os, openai, json

load_dotenv()
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

messages = []

In [3]:
FUNCTION_MAP = {
    'get_popular_movies': get_popular_movies,
    'get_movie_details': get_movie_details,
    'get_movie_credits': get_movie_credits,
    "get_similar_movies": get_similar_movies
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "A function to get the popular movies.",
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "A function to get the details of movie by movie ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The ID of the movie to get details."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "A function to get casts and crew of the movie by movie ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The ID of the movie to get casts and crew."
                    },
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "A function to get similar movies of the movie by movie ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "desscription": "The ID of the movie to get similar movies",
                    },
                },
                "required": ["id"]
            }
        }
    }
]

In [4]:
from openai.types.chat import ChatCompletionMessage

def process_ai_res(message: ChatCompletionMessage):
    if message.tool_calls:
        # TOOL CALL
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [{
                "id": tool_call.id,
                "type": "function",
                "function": {
                    "name": tool_call.function.name,
                    "arguments": tool_call.function.arguments
                }
            } for tool_call in message.tool_calls]
        })
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments
            print(f"Call Function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}
            
            function_to_run = FUNCTION_MAP[function_name]
            result = function_to_run(**arguments)
            print(f"Called {function_name} with {arguments}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result
            })
        call_ai()
    else:
        messages.append({
            "role": "assistant",
            "content": message.content
        })
        print(f"AI: {message.content}")

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS
    )

    process_ai_res(response.choices[0].message)

In [5]:
while True:
    userMessage = input("Enter a user message...")
    if userMessage == 'quit' or userMessage == 'q':
        print("Chat Finished >>>>>>>>")
        print(messages)
        break
    else:
        print(f"User: {userMessage}")
        messages.append({
            "role": "user",
            "content": userMessage
        })
        call_ai()
        

User: 요즘 인기있는 영화
Call Function: get_popular_movies with {}
Called get_popular_movies with {}
AI: 최근 인기 있는 영화 목록입니다:

1. **Mercy**
   - **개요:** 가까운 미래, 한 형사가 부인 살해 혐의로 재판을 받고 있다. 그는 자신의 무죄를 입증하기 위해 이전에 지지했던 AI 판사에게 90분을 주어진다.
   - **개봉일:** 2026-01-20
   - **평점:** 7.12
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **Shelter**
   - **개요:** 자발적으로 유배된 남자가 폭풍우에서 젊은 소녀를 구하면서 겪는 사건들로 인해 과거의 적들로부터 그녀를 보호해야 한다.
   - **개봉일:** 2026-01-28
   - **평점:** 7.0
   - ![Shelter](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

3. **The Orphans**
   - **개요:** 입양소에서 친구였던 두 남자가 서로의 과거와 직면하게 되는 이야기.
   - **개봉일:** 2025-08-20
   - **평점:** 6.11
   - ![The Orphans](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - **개요:** 섬에서 조용한 삶을 사는 한 전직 해적이 복수를 위해 과거의 적과 대면하게 된다.
   - **개봉일:** 2026-02-17
   - **평점:** 5.7
   - ![The Bluff](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJyAisX8TWjov.jpg)

5. **A Woman Scorned**
   - **개요: